In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
os.environ["LANG"] = "en_US.UTF-8"
os.environ["LANGUAGE"] = "en"

In [ ]:
# Run in Colab
!pip install transformers torch torchaudio scipy gradio accelerate sentencepiece openai-whisper
!apt-get install -qq -y ffmpeg

import torch
from transformers import pipeline, MusicgenForConditionalGeneration, AutoProcessor
import scipy.io.wavfile
import gradio as gr
import numpy as np

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 19.4 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for openai-whisper: filename=openai_whisper-20250625-py3-none-any.whl size=803979 sha256=f43de91dd35930c762f451919f2a2d2b030027e8e16d065459b2d8163c163930
  Stored in directory: /root/.cache/pip/wheels/61/d2/20/09ec9bef734d126cba375b15898010b6cc28578d8afdde5869
Successfully built openai-whisper


In [ ]:
import whisper
import re
import torch
import gc
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.cuda.empty_cache()
gc.collect()

lang_detector = pipeline(
    "text-classification",
    model="papluca/xlm-roberta-base-language-detection",
)

print("Loading translation model...")
nllb_tokenizer = AutoTokenizer.from_pretrained("facebook/nllb-200-distilled-600M")
nllb_model = AutoModelForSeq2SeqLM.from_pretrained(
    "facebook/nllb-200-distilled-600M",
    dtype=torch.float16,
).to(device)

NLLB_LANG_CODES = {
    "zh": "zho_Hans", "ja": "jpn_Jpan", "ko": "kor_Hang",
    "es": "spa_Latn", "fr": "fra_Latn", "de": "deu_Latn",
    "ar": "arb_Arab", "ru": "rus_Cyrl", "it": "ita_Latn",
    "pt": "por_Latn", "hi": "hin_Deva", "tr": "tur_Latn",
    "vi": "vie_Latn", "th": "tha_Thai", "id": "ind_Latn",
    "pl": "pol_Latn", "nl": "nld_Latn",
}

def translate_to_english(text):
    result = lang_detector(text[:512])[0]
    lang = result["label"]
    if lang == "en":
        return text, "en"
    nllb_code = NLLB_LANG_CODES.get(lang)
    if not nllb_code:
        return text, lang
    inputs = nllb_tokenizer(
        text, return_tensors="pt", truncation=True, max_length=512
    ).to(device)
    translated = nllb_model.generate(
        **inputs,
        forced_bos_token_id=nllb_tokenizer.convert_tokens_to_ids("eng_Latn"),
        max_length=512,
    )
    return nllb_tokenizer.decode(translated[0], skip_special_tokens=True), lang

print("✅ Translation model ready!")

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: papluca/xlm-roberta-base-language-detection
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/502 [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/2.46G [00:00<?, ?B/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

✅ Translation model ready!


In [ ]:
print("Loading models...")

emotion_classifier = pipeline(
    "text-classification",
    model="SamLowe/roberta-base-go_emotions",
    top_k=None
)

musicgen_processor = AutoProcessor.from_pretrained("facebook/musicgen-small")
musicgen_model = MusicgenForConditionalGeneration.from_pretrained("facebook/musicgen-small").to(device)
whisper_model = whisper.load_model("base", device=device)

# Move to GPU if available
device = "cuda" if torch.cuda.is_available() else "cpu"
musicgen_model.to(device)
# Whisper 语音识别
whisper_model = whisper.load_model("base", device=device)
print(f"✅ Models loaded on {device}!")

Loading models...


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: SamLowe/roberta-base-go_emotions
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

Loading weights:   0%|          | 0/611 [00:00<?, ?it/s]

MusicgenForConditionalGeneration LOAD REPORT from: facebook/musicgen-small
Key                                           | Status     |  | 
----------------------------------------------+------------+--+-
decoder.model.decoder.embed_positions.weights | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/224 [00:00<?, ?B/s]

100%|████████████████████████████████████████| 139M/139M [00:00<00:00, 178MiB/s]


✅ Models loaded on cuda!


In [ ]:
# 新增：GoEmotions 28个标签 → 音乐簇映射
EMOTION_CLUSTER = {
    "joy": "joy", "amusement": "joy",
    "excitement": "excitement",
    "optimism": "hope", "pride": "triumph",
    "love": "love", "desire": "longing",
    "caring": "warmth", "gratitude": "warmth",
    "admiration": "triumph", "approval": "warmth",
    "sadness": "sadness", "grief": "sadness",
    "disappointment": "sadness", "remorse": "sadness",
    "embarrassment": "sadness",
    "anger": "anger", "annoyance": "anger", "disapproval": "anger",
    "fear": "fear", "nervousness": "fear",
    "surprise": "surprise", "realization": "surprise",
    "disgust": "disgust",
    "neutral": "calm", "confusion": "calm", "relief": "calm",
    "curiosity": "curiosity",
    "nostalgia": "longing",
}

# 把原来的 EMOTION_TO_MUSIC 整个替换成下面这个（扩展到12个情绪簇）
EMOTION_TO_MUSIC = {
    "joy": [
        "uplifting pop music with bright piano melody and cheerful rhythm, major key, 120 bpm",
        "happy acoustic guitar with light percussion, sunny atmosphere",
        "gentle ukulele with bells, warm and breezy",
    ],
    "excitement": [
        "high-energy electronic dance music, pulsing bass, soaring synths, 140 bpm",
        "energetic indie pop with driving guitar riff and punchy drums",
        "upbeat funk groove with brass stabs, lively",
    ],
    "hope": [
        "inspiring orchestral swell with strings and french horn, heroic, building",
        "cinematic piano with warm pads, optimistic melody, 100 bpm",
        "gentle acoustic with subtle strings, quiet dawn atmosphere",
    ],
    "triumph": [
        "epic orchestral fanfare with full brass, choir, thunderous timpani",
        "powerful rock anthem with soaring guitar solo, triumphant energy",
        "proud march with brass and snare, confident and majestic",
    ],
    "love": [
        "romantic piano and strings with warm melody, tender and intimate, 72 bpm",
        "soft jazz ballad with saxophone, warm and sensual atmosphere",
        "acoustic love song with gentle vocals, heartfelt and emotional",
    ],
    "warmth": [
        "cozy acoustic guitar and piano duet, warm pads, 80 bpm, comforting",
        "soft chamber music with pizzicato strings, affectionate",
        "ambient folk with soft acoustic instruments, safe atmosphere",
    ],
    "longing": [
        "bittersweet piano with muted strings, nostalgic melody, 68 bpm",
        "melancholic acoustic guitar with reverb, introspective",
        "ambient pads with solo violin, ethereal, fading echoes",
    ],
    "sadness": [
        "melancholic piano ballad with soft strings, slow tempo, minor key",
        "emotional violin solo with ambient pads, contemplative mood",
        "gentle acoustic guitar with rain sounds, introspective atmosphere",
    ],
    "anger": [
        "intense rock music with heavy electric guitar and powerful drums, aggressive",
        "dramatic orchestral music with brass and percussion, epic battle theme",
        "industrial electronic music with distorted bass and intense rhythm",
    ],
    "fear": [
        "dark ambient music with low drones and eerie sounds, mysterious tension",
        "suspenseful orchestral strings with dissonant chords, horror atmosphere",
        "ominous electronic soundscape with deep bass, unsettling mood",
    ],
    "surprise": [
        "quirky electronic music with unexpected changes and playful melody",
        "upbeat jazz with spontaneous improvisations and dynamic rhythm",
        "whimsical orchestral music with pizzicato strings, playful atmosphere",
    ],
    "disgust": [
        "dark dissonant orchestral with grinding brass, discordant strings",
        "industrial noise with heavy distortion, irregular rhythms",
        "minor key drone with chromatic melody, oppressive",
    ],
    "calm": [
        "peaceful ambient with soft pads, gentle piano, 55 bpm, meditative",
        "minimalist acoustic guitar with long reverb, serene",
        "nature soundscape with soft sine tones, tranquil",
    ],
    "curiosity": [
        "playful pizzicato strings with music-box melody, inquisitive",
        "acoustic folk with bouncy rhythm, exploratory",
        "ambient electronic with gentle arpeggios, wonder",
    ],
}

def get_music_prompt(emotion_label, confidence):
    cluster = EMOTION_CLUSTER.get(emotion_label, "calm")   # ← 加这行
    prompts = EMOTION_TO_MUSIC.get(cluster, EMOTION_TO_MUSIC['calm'])  # ← 改这行
    if confidence > 0.8:
        return prompts[0]
    elif confidence > 0.5:
        return prompts[1] if len(prompts) > 1 else prompts[0]
    else:
        return prompts[-1]

In [ ]:
def parse_dialogue(dialogue_text):

    pairs = []
    for line in dialogue_text.splitlines():
        line = line.strip()
        if not line:
            continue
        m = re.match(r"^(M|F)\s*:\s*(.+)$", line, flags=re.IGNORECASE)
        if m:
            pairs.append((m.group(1).upper(), m.group(2).strip()))
        else:
            pairs.append(("N", line))
    return pairs

def emotion_vector(text):

    em = emotion_classifier(text)[0]
    return {e["label"]: float(e["score"]) for e in em}

def aggregate_emotion_dialogue(dialogue_text, alpha=0.6, topk=3):

    pairs = parse_dialogue(dialogue_text)
    vecs = [emotion_vector(utt) for _, utt in pairs]
    labels = list(vecs[0].keys())
    T = len(vecs)

    weights = np.array([np.exp(-alpha * (T-1-t)) for t in range(T)], dtype=np.float64)
    weights = weights / (weights.sum() + 1e-12)

    agg = {lab: 0.0 for lab in labels}
    for w, v in zip(weights, vecs):
        for lab in labels:
            agg[lab] += w * v.get(lab, 0.0)

    s = sum(agg.values())
    if s > 0:
        for k in agg:
            agg[k] /= s

    ranked = sorted(agg.items(), key=lambda x: x[1], reverse=True)
    return agg, ranked[:topk], ranked

In [ ]:
def build_composite_prompt(top3, intensity):
    """
    top3: [('sadness',0.42), ('love',0.31), ('joy',0.12)]
    intensity: top1
    """
    primary = top3[0][0]
    secondary = [x[0] for x in top3[1:]]

    # intensity -> tempo / dynamics
    if intensity > 0.80:
        tempo = "115-135 bpm"
        dynamics = "high energy, strong beat"
    elif intensity > 0.55:
        tempo = "90-115 bpm"
        dynamics = "moderate energy"
    else:
        tempo = "65-90 bpm"
        dynamics = "soft, gentle dynamics"

    base = get_music_prompt(primary, intensity)

    blend = []
    if "love" in secondary: blend.append("warm, intimate")
    if "sadness" in secondary: blend.append("melancholic undertone")
    if "joy" in secondary: blend.append("hopeful lift")
    if "anger" in secondary: blend.append("tense edge")
    if "fear" in secondary: blend.append("mysterious tension")
    if "surprise" in secondary: blend.append("unexpected twists")
    if "excitement" in secondary: blend.append("electrifying energy")
    if "hope" in secondary: blend.append("optimistic lift")
    if "triumph" in secondary: blend.append("majestic pride")
    if "warmth" in secondary: blend.append("tender comfort")
    if "longing" in secondary: blend.append("wistful nostalgia")
    if "disgust" in secondary: blend.append("dark dissonance")
    if "calm" in secondary: blend.append("peaceful stillness")
    if "curiosity" in secondary: blend.append("inquisitive wonder")
    blend_text = ", ".join(blend) if blend else "balanced emotional nuance"

    return f"{base}, {blend_text}, {tempo}, {dynamics}, cinematic background score"


In [ ]:
def score_audio(audio, sr):

    audio = np.asarray(audio, dtype=np.float32).squeeze()
    rms = float(np.sqrt(np.mean(audio**2) + 1e-12))
    clip_ratio = float(np.mean(np.abs(audio) > 0.99))

    spec = np.abs(np.fft.rfft(audio))
    freqs = np.fft.rfftfreq(audio.size, d=1.0/sr)
    low = spec[(freqs >= 20) & (freqs < 200)].sum()
    mid = spec[(freqs >= 200) & (freqs < 2000)].sum()
    high = spec[(freqs >= 2000) & (freqs < 8000)].sum()
    total = low + mid + high + 1e-12
    low_ratio = float(low / total)
    high_ratio = float(high / total)

    score = 0.0
    if 0.02 <= rms <= 0.18:
        score += 1.0
    else:
        score -= abs(rms - 0.08) * 5.0

    score -= clip_ratio * 50.0
    score -= max(0.0, low_ratio - 0.55) * 5.0
    score -= max(0.0, high_ratio - 0.35) * 5.0

    meta = {"rms": rms, "clip_ratio": clip_ratio, "low_ratio": low_ratio, "high_ratio": high_ratio}
    return score, meta


def generate_dialogue_bgm(dialogue_text, duration=10, temperature=1.0, n_candidates=4):

    _, top3, ranked = aggregate_emotion_dialogue(dialogue_text, alpha=0.6, topk=3)
    intensity = top3[0][1]
    music_prompt = build_composite_prompt(top3, intensity)

    inputs = musicgen_processor(text=[music_prompt], padding=True, return_tensors="pt").to(device)
    max_new_tokens = int(duration * 50)

    best = None
    best_score = -1e9
    best_meta = None

    for _ in range(n_candidates):
        with torch.no_grad():
            audio_values = musicgen_model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=True,
                temperature=temperature,
                guidance_scale=3.0
            )

        sr = musicgen_model.config.audio_encoder.sampling_rate
        audio = audio_values[0, 0].cpu().numpy()
        sc, meta = score_audio(audio, sr)

        if sc > best_score:
            best_score = sc
            best = (sr, audio)
            best_meta = meta

    significant = [x for x in ranked if x[1] > 0.10]
    summary = ", ".join([f"{lab}: {sc:.1%}" for lab, sc in significant]) if significant else \
              ", ".join([f"{lab}: {sc:.1%}" for lab, sc in ranked[:6]])

    return best, top3, intensity, summary, music_prompt, best_score, best_meta


In [ ]:
import re

IGNORE_LABELS = {'neutral', 'approval', 'realization'}

def split_sentences(text):
    parts = re.split(r'[，。！？,!?\.]+', text)
    return [p.strip() for p in parts if p.strip()]

def generate_emotion_music(text_input, duration=10, temperature=1.0):
    # ... 原来的代码不动
    """
    Generate music based on text emotion

    Args:
        text_input: User's text (journal entry, story, tweet, etc.)
        duration: Music length in seconds (5-30 recommended)
        temperature: Creativity (0.8 = conservative, 1.2 = creative)
    """

    text_input, detected_lang = translate_to_english(text_input)
    if detected_lang != "en":
      print(f"🌐 Translated from: {detected_lang}")

# Step 1: Analyze emotion
    sentences = split_sentences(text_input)

    if len(sentences) <= 1:
        raw = emotion_classifier(text_input)[0]                          # ← 保留，单句用这个
        emotions = [e for e in raw if e['label'] not in IGNORE_LABELS]
    else:
        all_scores = []
        for i, sent in enumerate(sentences):
            weight = 1.0 + i * 0.5
            scores = emotion_classifier(sent)[0]
            all_scores.append((weight, {e['label']: e['score'] for e in scores}))

        total_weight = sum(w for w, _ in all_scores)
        merged = {}
        for w, score_dict in all_scores:
            for label, score in score_dict.items():
                if label in IGNORE_LABELS:
                    continue
                merged[label] = merged.get(label, 0) + score * w / total_weight

        total = sum(merged.values())
        merged = {k: v / total for k, v in merged.items()}
        emotions = [{'label': k, 'score': v} for k, v in merged.items()]

    # admiration 补偿
    for e in emotions:
        if e['label'] == 'admiration' and e['score'] > 0.8:
            joy_score = e['score'] * 0.5
            e['score'] = e['score'] * 0.5
            joy_entry = next((x for x in emotions if x['label'] == 'joy'), None)
            if joy_entry:
                joy_entry['score'] += joy_score
            else:
                emotions.append({'label': 'joy', 'score': joy_score})
            break

    top_emotion = max(emotions, key=lambda x: x['score'])
    emotion_label = top_emotion['label']
    confidence = top_emotion['score']

    print(f"🎭 Detected: {emotion_label.upper()} ({confidence:.2%} confidence)")

    # Step 2: Get all significant emotions (>10%)
    significant_emotions = [e for e in emotions if e['score'] > 0.2]
    emotion_summary = ", ".join([f"{e['label']}: {e['score']:.1%}" for e in significant_emotions])

    # Step 3: Generate music prompt
    music_prompt = get_music_prompt(emotion_label, confidence)
    print(f"🎵 Generating: {music_prompt}")

    # Step 4: Generate music
    inputs = musicgen_processor(
        text=[music_prompt],
        padding=True,
        return_tensors="pt"
    ).to(device)

    # Calculate tokens for desired duration (50 tokens ≈ 1 second)
    max_new_tokens = int(duration * 50)

    with torch.no_grad():
        audio_values = musicgen_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=temperature,
            guidance_scale=3.0  # How closely to follow prompt (2-4 recommended)
        )

    # Step 5: Convert to audio
    sampling_rate = musicgen_model.config.audio_encoder.sampling_rate
    audio_data = audio_values[0, 0].cpu().numpy()

    return (sampling_rate, audio_data), emotion_label, confidence, emotion_summary, music_prompt

In [ ]:
def translate_text(text, target_lang):
    if target_lang == "en":
        return text

    nllb_code = NLLB_LANG_CODES.get(target_lang)
    if not nllb_code:
        return text

    inputs = nllb_tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=512,
        src_lang="eng_Latn"
    ).to(device)

    translated_ids = nllb_model.generate(
        **inputs,
        forced_bos_token_id=nllb_tokenizer.convert_tokens_to_ids(nllb_code),
        max_length=512,
    )

    return nllb_tokenizer.decode(translated_ids[0], skip_special_tokens=True)


def gradio_interface(text, duration, temperature):
    detected = lang_detector(text[:512])[0]
    input_lang = detected["label"]

    (sr, audio), emotion, conf, summary, prompt = generate_emotion_music(
        text, duration, temperature
    )

    full_text = (
        f"Dominant Emotion: {emotion.upper()} ({conf:.1%} confidence)\n"
        f"All Emotions: {summary}\n"
        f"Music Style: {prompt}"
    )

    emotion_text = translate_text(full_text, input_lang)
    return (sr, audio), emotion_text


def gradio_voice(audio_path, duration, temperature):
    if audio_path is None:
        return None, "", "⚠️ Please record or upload an audio file first."
    result = whisper_model.transcribe(audio_path, fp16=(device == "cuda"))
    text = result["text"].strip()
    lang = result.get("language", "unknown")
    print(f"🌐 检测语言: {lang}  📝 转录: {text}")

    audio_result, analysis_text = gradio_interface(text, duration, temperature)
    return audio_result, text, analysis_text  # ← 多返回一个转录文本

In [ ]:
gr.close_all()

with gr.Blocks(title="🎭 Emotion-Driven Music Generator") as demo:
    gr.Markdown("# 🎭 Emotion-Driven Music Generator\nMultilingual · Voice Input")

    with gr.Tabs():
        with gr.Tab("📝 Text Input"):
            txt = gr.Textbox(
                label="Enter text (any language)",
                placeholder="Example: I finally got my dream job! / 今天终于实现了梦想！",
                lines=5
            )
            with gr.Row():
                dur = gr.Slider(minimum=5, maximum=30, value=10, step=1, label="Music Duration (seconds)")
                tmp = gr.Slider(minimum=0.7, maximum=1.3, value=1.0, step=0.1, label="Creativity (Temperature)")
            btn = gr.Button("🎵 Generate Music", variant="primary")
            with gr.Row():
                aud = gr.Audio(label="Generated Music 🎵", type="numpy")
                inf = gr.Markdown(label="Emotion Analysis 🎭")
            gr.Examples(
                examples=[
                    ["I finally achieved my lifelong dream today! Everything feels perfect.", 10, 1.0],
                    ["I miss the old days when everything was simpler. Time flies too fast.", 12, 1.0],
                    ["今天终于实现了我的梦想，感觉整个世界都在为我庆祝！", 10, 1.0],
                    ["最近心情很低落，总是想起过去的美好时光，感觉很孤独。", 12, 0.9],
                    ["Tengo la gran fortuna de vivir en un país pacífico.", 10, 1.0]
                ],
                inputs=[txt, dur, tmp]
            )
            btn.click(fn=gradio_interface, inputs=[txt, dur, tmp], outputs=[aud, inf])

        with gr.Tab("🎙️ Voice Input"):
            gr.Markdown("Upload an audio file or record from microphone. Supports any language.")
            mic = gr.Audio(
                sources=["microphone", "upload"],
                type="filepath",
                label="🎤 Record or Upload Audio"
            )
            with gr.Row():
                v_dur = gr.Slider(minimum=5, maximum=30, value=10, step=1, label="Music Duration (seconds)")
                v_tmp = gr.Slider(minimum=0.7, maximum=1.3, value=1.0, step=0.1, label="Creativity (Temperature)")
            v_btn = gr.Button("🎙️ Transcribe & Generate Music", variant="primary")

            # 转录文本显示框 ← 新增
            v_txt = gr.Textbox(
                label="📝 Transcribed Text",
                interactive=False,
                lines=3
            )
            with gr.Row():
                v_aud = gr.Audio(label="Generated Music 🎵", type="numpy")
                v_inf = gr.Markdown(label="Emotion Analysis 🎭")

            v_btn.click(
                fn=gradio_voice,
                inputs=[mic, v_dur, v_tmp],
                outputs=[v_aud, v_txt, v_inf]  # ← 多一个 v_txt 输出
            )

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://7740a262fda3dba97f.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
